[![img/pythonista.png](img/pythonista.png)](https://www.pythonista.io)

# La lógica del contraste de hipótesis.

**py401 · EDA Práctico** · Pythonista® · Instructor: José Luis Chiquete · Módulo III: *Inferencia estadística*

> 📝 **Versión del alumno.** Las celdas marcadas con `# TU CÓDIGO AQUÍ` y las de *Tu respuesta* son para ti.

En este *notebook* se presentará la lógica del contraste de hipótesis: H0 y H1, el valor *p*, los errores tipo I y II y la potencia de una prueba.

## 🧭 Cómo usar este *notebook*.

* Ejecuta las celdas **en orden** (`Shift + Enter`).
* 🟢 **Ejercicio básico:** lo que todos deben dominar.
* 🔴 **Reto (opcional):** para quien termina antes o quiere profundizar. Si no da tiempo en clase, se puede resolver como práctica.
* Las celdas `# TU CÓDIGO AQUÍ` y *✍️ Tu respuesta* son para ti.

## 📚 Referencias.

py311 no cubre inferencia, así que la referencia de la API es la documentación oficial: [*SciPy* · `scipy.stats`](https://docs.scipy.org/doc/scipy/reference/stats.html) (`ttest_ind`, `mannwhitneyu`, `permutation_test`, `chi2_contingency`, `f_oneway`, `levene`), [*statsmodels* · pruebas de proporciones](https://www.statsmodels.org/stable/generated/statsmodels.stats.proportion.proportions_ztest.html), [*statsmodels* · `pairwise_tukeyhsd`](https://www.statsmodels.org/stable/generated/statsmodels.stats.multicomp.pairwise_tukeyhsd.html) y [*statsmodels* · `anova_oneway`](https://www.statsmodels.org/stable/generated/statsmodels.stats.oneway.anova_oneway.html).

### Preparación del entorno.

* La siguiente celda importará las bibliotecas y las funciones de *SciPy* y *statsmodels* que se usarán, y definirá `DATA` y la semilla `rng`.

In [ ]:
import os
import warnings

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

DATA = next(os.path.join(p, "data") for p in (os.curdir, os.pardir)
            if os.path.isfile(os.path.join(p, "data", "penguins.csv")))
pd.set_option("display.max_columns", 20)
sns.set_theme(style="whitegrid")
rng = np.random.default_rng(401)
from scipy import stats
from statsmodels.stats.multicomp import pairwise_tukeyhsd
from statsmodels.stats.multitest import multipletests
from statsmodels.stats.oneway import anova_oneway
from statsmodels.stats.power import TTestIndPower
from statsmodels.stats.proportion import proportions_ztest

* La siguiente celda leerá `penguins_limpio.csv` (o lo regenerará si no existe), `titanic.csv` y `tips.csv`, y creará `tip_pct`.

In [ ]:
ruta_limpio = os.path.join(DATA, "penguins_limpio.csv")
medidas = ["bill_length_mm", "bill_depth_mm", "flipper_length_mm", "body_mass_g"]
if not os.path.isfile(ruta_limpio):
    (
        pd.read_csv(os.path.join(DATA, "penguins.csv"))
        .assign(sex=lambda d: d["sex"].str.title())
        .dropna(subset=medidas, how="all")
        .to_csv(ruta_limpio, index=False)
    )
pinguinos = pd.read_csv(ruta_limpio)
titanic = pd.read_csv(os.path.join(DATA, "titanic.csv"))
propinas = pd.read_csv(os.path.join(DATA, "tips.csv")).assign(tip_pct=lambda d: 100 * d["tip"] / d["total_bill"])
pinguinos.shape, titanic.shape, propinas.shape

---
## 1. Repaso (10 min).

En la sesión 5, un IC del 95% para la diferencia de masa media entre machos y hembras Gentoo fue de unos 700 a 910 g: el 0 quedó muy lejos. Una prueba de hipótesis responde la pregunta complementaria: **si en la población no hubiera diferencia, ¿qué tan raro sería observar una diferencia como esta?** IC y prueba se complementan; el IC además informa la **magnitud**.

---
## 2. La lógica del contraste (20 min).

| Concepto | Significado |
|---|---|
| H0 (hipótesis nula) | "No hay efecto": diferencia 0, independencia, medias iguales |
| H1 (alternativa) | Hay diferencia (bilateral) o va en una dirección (unilateral, decidida **antes** de ver los datos) |
| Valor *p* | Probabilidad, **si H0 fuera cierta**, de obtener un resultado al menos tan extremo como el observado |
| α (nivel de significancia) | Umbral fijado de antemano (habitualmente 0.05) |
| Error tipo I | Rechazar H0 siendo cierta (probabilidad α) |
| Error tipo II | No rechazar H0 siendo falsa (probabilidad β) |
| Potencia | 1 − β: probabilidad de detectar un efecto que sí existe |

**Advertencia:** el valor *p* **no** es la probabilidad de que H0 sea cierta, no mide el tamaño del efecto y *p* > 0.05 **no** demuestra que no haya diferencia.

**Concepto: cómo se construye una prueba.** Toda prueba de hipótesis sigue los mismos cuatro pasos:

1. Plantear H0 y H1 **antes** de ver los resultados.
2. Elegir un **estadístico de prueba** que mida qué tan lejos están los datos de lo que predice H0. Por ejemplo, para dos medias, $t = \dfrac{\bar{x}_A - \bar{x}_B}{EE(\bar{x}_A - \bar{x}_B)}$: la diferencia observada medida en errores estándar.
3. Obtener su **distribución bajo H0** (con una fórmula, como la *t* de Student, o por simulación, como la permutación).
4. Calcular el valor *p*: $p = P(|T| \geq |t_{obs}| \mid H_0)$. Si $p < \alpha$, se rechaza H0.

Si H0 es cierta, el valor *p* se distribuye **uniforme** entre 0 y 1: hay 5% de probabilidad de que salga menor que 0.05 por puro azar. Eso es exactamente el error tipo I, y es lo que explota el *p-hacking*.

**Ejemplo: comparaciones múltiples y *p-hacking*.** Se simularán 1,000 "estudios" en los que se comparan 20 variables entre dos grupos **idénticos** (H0 es cierta en todas). ¿En qué fracción de estudios aparece al menos un *p* < 0.05?

* La siguiente celda simulará los 1,000 estudios con 20 pruebas *t* cada uno, de forma vectorizada.

In [ ]:
grupo_a = rng.normal(size=(1000, 20, 30))
grupo_b = rng.normal(size=(1000, 20, 30))
valores_p = stats.ttest_ind(grupo_a, grupo_b, axis=2).pvalue
float((valores_p < 0.05).any(axis=1).mean())

Cerca de 64% de los estudios "encuentra algo" (1 − 0.95²⁰ ≈ 0.64) sin que exista ningún efecto. Probar muchas variables, subgrupos o versiones de un análisis y reportar solo lo significativo es *p-hacking*. **Decisiones:** definir las preguntas antes de mirar los resultados, reportar todas las pruebas hechas y, si son muchas, corregir con Holm o Bonferroni (`multipletests`).

* La siguiente celda aplicará la corrección de Holm a las 20 pruebas de cada estudio y calculará en qué fracción de estudios queda algún hallazgo.

In [ ]:
algun_hallazgo_holm = np.array([multipletests(p_estudio, alpha=0.05, method="holm")[0].any() for p_estudio in valores_p])
float(algun_hallazgo_holm.mean())

Con la corrección, la fracción de estudios con algún falso hallazgo vuelve a quedar alrededor de α = 5%. El costo es menos potencia para cada prueba individual: por eso conviene hacer pocas pruebas, bien elegidas.

**Interpretación:** con Holm, solo ≈ 5.6% de los estudios conserva algún falso hallazgo. La corrección de Holm ordena los $m$ valores *p* de menor a mayor y compara el *k*-ésimo con $\alpha / (m - k + 1)$; Bonferroni, más conservadora, compara todos con $\alpha / m$.

### 2.1 Potencia: ¿cuántos datos hacen falta?

La potencia depende del tamaño del efecto, de *n* y de α. `TTestIndPower` de *statsmodels* resuelve cualquiera de ellos dados los demás. El tamaño del efecto se expresa con la **d de Cohen** (diferencia de medias en unidades de desviación estándar).

**Concepto: d de Cohen.** Es la diferencia de medias en unidades de la desviación estándar combinada:

$$d = \frac{\bar{x}_A - \bar{x}_B}{s_p}, \qquad s_p = \sqrt{\frac{(n_A - 1)s_A^2 + (n_B - 1)s_B^2}{n_A + n_B - 2}}$$

Al no tener unidades, permite comparar efectos entre variables distintas y es el insumo de los cálculos de potencia.

* La siguiente celda calculará el *n* por grupo necesario para detectar d = 0.5 con 80% de potencia, y la potencia que se tendría con d = 0.3 y 30 por grupo.

In [ ]:
potencia = TTestIndPower()
(round(potencia.solve_power(effect_size=0.5, alpha=0.05, power=0.8), 1),
 round(float(potencia.power(effect_size=0.3, nobs1=30, alpha=0.05)), 3))

Se necesitan unos 64 casos por grupo para un efecto mediano; con 30 por grupo y un efecto pequeño (d = 0.3), solo se detectaría 1 de cada 5 veces. **Consecuencia:** en un estudio pequeño, "no significativo" casi no informa nada.

**💡 Ideas clave de la sección 2:**
* El valor *p* mide qué tan sorprendentes serían los datos **si H0 fuera cierta**; no es la probabilidad de H0.
* Con muchas pruebas aparecen falsos hallazgos: se planean pocas pruebas y se corrige (Holm).
* La potencia se planea antes: con *n* pequeño, un "no significativo" casi no informa.

### 🟢 Ejercicio 1 · ¿Qué dice un valor *p*?

**🎯 Objetivo:** distinguir la definición del valor *p* de sus lecturas erróneas más comunes. **Qué se espera aprender:** a no confundir *p* con la probabilidad de H0, con la magnitud del efecto ni con una prueba de igualdad.

Una prueba compara la masa de dos colonias y obtiene *p* = 0.03. Indica si cada afirmación es correcta: (a) "hay 3% de probabilidad de que H0 sea cierta"; (b) "si las colonias no difirieran, una diferencia así de grande o mayor ocurriría en torno a 3% de las veces"; (c) "la diferencia es grande"; (d) "con *p* = 0.30 se habría demostrado que las colonias son iguales".

**✍️ Tu respuesta:**

_(escribe aquí)_

---
## ✅ Cierre.

**Práctica (sin calificación, en Classroom):** terminar los ejercicios 🟢 pendientes de este *notebook* y, si es posible, un reto 🔴.

**Siguiente *notebook*:** `20_pruebas_para_dos_grupos.ipynb`.

<p style="text-align: center"><a rel="license" href="http://creativecommons.org/licenses/by/4.0/"><img alt="Licencia Creative Commons" style="border-width:0" src="https://i.creativecommons.org/l/by/4.0/80x15.png" /></a><br />Esta obra está bajo una <a rel="license" href="http://creativecommons.org/licenses/by/4.0/">Licencia Creative Commons Atribución 4.0 Internacional</a>.</p>
<p style="text-align: center">&copy; José Luis Chiquete Valdivieso. 2017-2026.</p>